In [19]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q kaggle

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import os, json, shutil, subprocess, tarfile, time, glob
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from kaggle.api.kaggle_api_extended import KaggleApi

os.chdir('/content')

RAW   = '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/raw'
USER  = 'minhtri2475'
PART  = 1                              # <<< tăng 1, 2, 3... mỗi lần chạy
N_THIS_RUN = 22                        # lần đầu 22 để gom lại 0001-0022; từ PART=2 đặt 300
SLUG  = f'lidc-idri-raw-p{PART:02d}'
DATASET = f'{USER}/{SLUG}'
STAGE = '/content/stage/upload'
MANIFEST = 'done_patients.txt'                                   # manifest của riêng phần này (nằm trong dataset)
DRIVE_MANIFEST = os.path.join(os.path.dirname(RAW), MANIFEST)    # manifest tổng (nằm trên Drive)

api = KaggleApi(); api.authenticate()

def sh(*a):
    return subprocess.run(a, capture_output=True, text=True)

def ds_status():
    r = sh('kaggle', 'datasets', 'status', DATASET)
    if r.returncode != 0:
        return ''
    lines = [l.strip().lower() for l in r.stdout.splitlines()
             if l.strip() and not l.strip().lower().startswith('warning')]
    return lines[-1] if lines else ''

def dataset_exists():
    return ds_status() in ('ready', 'pending', 'processing')

def get_manifest():
    tmp = '/content/tmp_manifest'
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    sh('kaggle', 'datasets', 'download', '-d', DATASET, '-f', MANIFEST, '-p', tmp, '--force')
    for z in glob.glob(f'{tmp}/*.zip'):
        shutil.unpack_archive(z, tmp)
    f = f'{tmp}/{MANIFEST}'
    return set(open(f).read().split()) if os.path.exists(f) else None

def patients_on_kaggle():
    """Tên thư mục cấp 1 trên Kaggle (Kaggle tự giải nén .tar nên không còn đuôi .tar)."""
    names, token = set(), None
    while True:
        r = api.dataset_list_files(DATASET, page_token=token, page_size=200)
        names |= {f.name.split('/')[0] for f in r.files}
        token = getattr(r, 'next_page_token', None) or getattr(r, 'nextPageToken', None)
        if not token:
            break
    return names

def wait_for_manifest(expected, timeout=3600, step=20):
    bar = tqdm(total=timeout, desc='Kaggle xử lý', unit='s',
               bar_format='{desc}: {elapsed} | {postfix}')
    t0 = time.time()
    while time.time() - t0 < timeout:
        st = ds_status()
        bar.set_postfix_str(f'trạng thái: {st or "chưa thấy"}')
        bar.update(0)
        if st == 'ready' and get_manifest() == expected:
            bar.close()
            return True
        time.sleep(step)
    bar.close()
    return False

# 1) Đọc manifest tổng trên Drive, chọn mẫu cho lần này
done = set(open(DRIVE_MANIFEST).read().split()) if os.path.exists(DRIVE_MANIFEST) else set()
patients = sorted(d for d in os.listdir(RAW) if d.startswith('LIDC-IDRI-'))
todo = [p for p in patients if p not in done][:N_THIS_RUN]
assert todo, 'Đã đẩy hết rồi!'
print(f'Đã đẩy trước đó: {len(done)} | Tổng {len(patients)} | lần này đẩy {len(todo)}')
print(f'Dataset: {DATASET} | Dải mẫu: {todo[0]} -> {todo[-1]}')
if len(todo) <= 30:
    print('Danh sách:', ', '.join(p.replace('LIDC-IDRI-', '') for p in todo))

# 2) Dataset PART này đã tồn tại chưa?
exists = dataset_exists()
already_uploaded = False
if exists:
    m = get_manifest()
    if m == set(todo):
        already_uploaded = True
        print('Dataset đã có đúng dữ liệu của lần này, bỏ qua upload và chuyển sang kiểm tra.')
    else:
        raise SystemExit(f'{DATASET} đã tồn tại nhưng nội dung khác dự kiến (hoặc chưa xử lý xong). '
                         f'Tăng PART lên để tạo phần mới, hoặc kiểm tra trên web.')

if not already_uploaded:
    # 3) Đóng gói
    os.makedirs(STAGE, exist_ok=True)
    for f in os.listdir(STAGE):
        if f.endswith('.part') or (f.endswith('.tar') and f[:-4] not in todo):
            os.remove(f'{STAGE}/{f}')

    def pack(p):
        out = f'{STAGE}/{p}.tar'
        if os.path.exists(out):
            return p
        tmp = out + '.part'
        with tarfile.open(tmp, 'w') as t:
            t.add(f'{RAW}/{p}', arcname=p)
        os.rename(tmp, out)
        return p

    with ThreadPoolExecutor(8) as ex:
        list(tqdm(ex.map(pack, todo), total=len(todo), desc='Đóng gói', unit='bn'))

    sizes = {p: os.path.getsize(f'{STAGE}/{p}.tar') for p in todo}
    print(f'Đã đóng gói {len(todo)} mẫu, tổng {sum(sizes.values())/1e9:.2f} GB')

    # 4) Upload: luôn tạo dataset mới, manifest chỉ chứa phần này
    open(f'{STAGE}/{MANIFEST}', 'w').write('\n'.join(sorted(todo)))
    json.dump({"title": SLUG, "id": DATASET, "licenses": [{"name": "CC0-1.0"}]},
              open(f'{STAGE}/dataset-metadata.json', 'w'))
    print(f'Đang upload {len(todo)} mẫu ({todo[0]} -> {todo[-1]})...')
    if subprocess.run(['kaggle', 'datasets', 'create', '-p', STAGE, '--dir-mode', 'skip']).returncode != 0:
        raise SystemExit(f'Upload lỗi. File vẫn còn ở {STAGE}, chạy lại cell là tiếp tục.')

# 5) Chờ Kaggle xử lý và kiểm tra thật sự
if not wait_for_manifest(set(todo)):
    raise SystemExit(f'Chưa thấy dữ liệu sẵn sàng sau 60 phút. Giữ nguyên {STAGE}. '
                     f'Kiểm tra trên web rồi chạy lại cell (không đổi PART).')

on_kaggle = patients_on_kaggle()
missing = sorted(set(todo) - on_kaggle)
if missing:
    raise SystemExit(f'Thiếu {len(missing)} mẫu trên Kaggle, ví dụ: {missing[:5]}. Giữ nguyên {STAGE}.')

# 6) Chỉ khi mọi thứ đã khớp mới ghi manifest Drive và dọn dẹp
with open(DRIVE_MANIFEST, 'w') as f:
    f.write('\n'.join(sorted(done | set(todo))))
shutil.rmtree(STAGE, ignore_errors=True)

total_done = len(done | set(todo))
print(f'Xong phần {PART}: {todo[0]} -> {todo[-1]} ({len(todo)} mẫu)')
print(f'Tổng đã đẩy {total_done}/{len(patients)} bệnh nhân, còn {len(patients)-total_done}. '
      f'Lần sau đổi PART = {PART+1}.')

OK


In [30]:
import os, json, shutil, subprocess, tarfile, time, glob
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from kaggle.api.kaggle_api_extended import KaggleApi

os.chdir('/content')

RAW   = '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/raw'
USER  = 'minhtri2475'
SLUG  = 'lidc-idri-raw'
DATASET = f'{USER}/{SLUG}'
N_THIS_RUN = 10                        # TEST: để 2. Test xong đổi thành 300 hoặc số bạn muốn
STAGE = '/content/stage/upload'
MANIFEST = 'done_patients.txt'

api = KaggleApi(); api.authenticate()

def sh(*a):
    return subprocess.run(a, capture_output=True, text=True)

def ds_status():
    """Trả về 'ready' / 'pending' / ... hoặc '' nếu dataset không tồn tại. Bỏ qua dòng Warning."""
    r = sh('kaggle', 'datasets', 'status', DATASET)
    if r.returncode != 0:
        return ''
    lines = [l.strip().lower() for l in r.stdout.splitlines()
             if l.strip() and not l.strip().lower().startswith('warning')]
    return lines[-1] if lines else ''

def dataset_exists():
    return ds_status() in ('ready', 'pending', 'processing')

def wait_ready(timeout=900, step=5):
    bar = tqdm(total=timeout, desc='Kaggle xử lý', unit='s',
               bar_format='{desc}: {elapsed} | {postfix}')
    t0 = time.time()
    while time.time() - t0 < timeout:
        st = ds_status()
        bar.set_postfix_str(f'trạng thái: {st or "chưa thấy"} (tối đa chờ {timeout//60} phút)')
        bar.update(0)
        if st == 'ready':
            bar.close()
            return True
        time.sleep(step)
    bar.close()
    return False

def get_manifest():
    tmp = '/content/tmp_manifest'
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    sh('kaggle', 'datasets', 'download', '-d', DATASET, '-f', MANIFEST, '-p', tmp, '--force')
    for z in glob.glob(f'{tmp}/*.zip'):
        shutil.unpack_archive(z, tmp)
    f = f'{tmp}/{MANIFEST}'
    return set(open(f).read().split()) if os.path.exists(f) else None

def patients_from_listing():
    """Chỉ dùng lần đầu khi chưa có manifest: suy ra bệnh nhân từ danh sách file."""
    names, token = set(), None
    while True:
        r = api.dataset_list_files(DATASET, page_token=token, page_size=200)
        names |= {f.name.split('/')[0] for f in r.files}
        token = getattr(r, 'next_page_token', None) or getattr(r, 'nextPageToken', None)
        if not token:
            break
    return names

# 1) Xem Kaggle đã có những bệnh nhân nào
exists = dataset_exists()
if exists:
    done = get_manifest()
    if done is None:
        print('Chưa có manifest, suy ra từ danh sách file...')
        done = patients_from_listing()
else:
    done = set()
print(f'Dataset tồn tại: {exists} | đã có trên Kaggle: {len(done)} bệnh nhân')

patients = sorted(d for d in os.listdir(RAW) if d.startswith('LIDC-IDRI-'))
todo = [p for p in patients if p not in done][:N_THIS_RUN]
print(f'Tổng {len(patients)} | còn lại {len(patients)-len(done)} | lần này đẩy {len(todo)}')
assert todo, 'Đã đẩy hết rồi!'
print(f'Dải mẫu lần này: {todo[0]} -> {todo[-1]}')
if len(todo) <= 30:
    print('Danh sách:', ', '.join(p.replace('LIDC-IDRI-', '') for p in todo))

# 2) Đóng gói
os.makedirs(STAGE, exist_ok=True)
for f in os.listdir(STAGE):
    if f.endswith('.part') or (f.endswith('.tar') and f[:-4] not in todo):
        os.remove(f'{STAGE}/{f}')

def pack(p):
    out = f'{STAGE}/{p}.tar'
    if os.path.exists(out):
        return p
    tmp = out + '.part'
    with tarfile.open(tmp, 'w') as t:
        t.add(f'{RAW}/{p}', arcname=p)
    os.rename(tmp, out)
    return p

with ThreadPoolExecutor(8) as ex:
    list(tqdm(ex.map(pack, todo), total=len(todo), desc='Đóng gói', unit='bn'))

sizes = {p: os.path.getsize(f'{STAGE}/{p}.tar') for p in todo}
total_gb = sum(sizes.values()) / 1e9
print(f'Đã đóng gói {len(todo)} mẫu, tổng {total_gb:.2f} GB')
if len(todo) <= 10:
    for p, s in sizes.items():
        print(f'  {p}: {s/1e6:.0f} MB')

# 3) Upload vào CÙNG dataset (kèm manifest)
expected = done | set(todo)
open(f'{STAGE}/{MANIFEST}', 'w').write('\n'.join(sorted(expected)))
json.dump({"title": SLUG, "id": DATASET, "licenses": [{"name": "CC0-1.0"}]},
          open(f'{STAGE}/dataset-metadata.json', 'w'))

msg = f'them {len(todo)} mau: {todo[0]} -> {todo[-1]}'
if exists:
    cmd = ['kaggle', 'datasets', 'version', '-p', STAGE, '-m', msg, '--dir-mode', 'skip']
else:
    cmd = ['kaggle', 'datasets', 'create', '-p', STAGE, '--dir-mode', 'skip']
print(f'Đang upload {len(todo)} mẫu ({todo[0]} -> {todo[-1]})...')
if subprocess.run(cmd).returncode != 0:
    raise SystemExit(f'Upload lỗi. File vẫn còn ở {STAGE}, chạy lại cell là tiếp tục.')

# 4) Chờ Kaggle xử lý xong: đợi tới khi manifest mới xuất hiện
def wait_for_new_version(expected, timeout=1800, step=20):
    bar = tqdm(total=timeout, desc='Kaggle xử lý', unit='s',
               bar_format='{desc}: {elapsed} | {postfix}')
    t0 = time.time()
    while time.time() - t0 < timeout:
        st = ds_status()
        bar.set_postfix_str(f'trạng thái: {st or "chưa thấy"}')
        bar.update(0)
        if st == 'ready':
            after = get_manifest()
            if after == expected:
                bar.close()
                return after
        time.sleep(step)
    bar.close()
    return None

after = wait_for_new_version(expected)
if after is None:
    raise SystemExit(f'Chưa thấy version mới sau 30 phút. Giữ nguyên {STAGE}. Kiểm tra trên web rồi chạy lại.')

ok = True
if done:
    first = api.dataset_list_files(DATASET, page_size=200).files
    ok = any(f.name.startswith(sorted(done)[0] + '/') for f in first)
if not ok:
    raise SystemExit(f'CẢNH BÁO: dữ liệu cũ có thể đã bị mất sau khi version. Giữ nguyên {STAGE}, chưa xoá.')

shutil.rmtree(STAGE)
print(f'Xong. Lần này đã đẩy: {todo[0]} -> {todo[-1]} ({len(todo)} mẫu)')
print(f'Dataset hiện có {len(expected)}/{len(patients)} bệnh nhân, còn {len(patients)-len(expected)} mẫu')

Dataset tồn tại: True | đã có trên Kaggle: 12 bệnh nhân
Tổng 1010 | còn lại 998 | lần này đẩy 10
Dải mẫu lần này: LIDC-IDRI-0013 -> LIDC-IDRI-0022
Danh sách: 0013, 0014, 0015, 0016, 0017, 0018, 0019, 0020, 0021, 0022


Đóng gói:   0%|          | 0/10 [00:00<?, ?bn/s]

Đã đóng gói 10 mẫu, tổng 0.95 GB
  LIDC-IDRI-0013: 68 MB
  LIDC-IDRI-0014: 61 MB
  LIDC-IDRI-0015: 113 MB
  LIDC-IDRI-0016: 78 MB
  LIDC-IDRI-0017: 126 MB
  LIDC-IDRI-0018: 69 MB
  LIDC-IDRI-0019: 163 MB
  LIDC-IDRI-0020: 120 MB
  LIDC-IDRI-0021: 80 MB
  LIDC-IDRI-0022: 71 MB
Đang upload 10 mẫu (LIDC-IDRI-0013 -> LIDC-IDRI-0022)...


Kaggle xử lý: 00:00 | 

SystemExit: CẢNH BÁO: dữ liệu cũ có thể đã bị mất sau khi version. Giữ nguyên /content/stage/upload, chưa xoá.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
